<a href="https://colab.research.google.com/github/shresta-28/TSA_DataScienceandAnalytics/blob/main/ev_strain_dataScience_TSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import seaborn as sns

ev_grid_og = pd.read_csv("ev_grid_dataset.csv")

In [26]:
ev_grid_working = ev_grid_og.copy()

def count_blanks(data):
  print("Blanks in Avg_Temp_F:", data["Avg_Temp_F"].isna().sum())
  print("Blanks in Public_Charger_Count:", data["Public_Charger_Count"].isna().sum())

  load_cols = ["Base_Load_MW", "Home_EV_Load_MW", "Public_EV_Load_MW",
              "EV_Charging_Load_MW", "Total_Load_MW"]
  print("Blanks in load columns (should be 0):\n", data[load_cols].isna().sum())

  blanks_before = data.isna().sum()
  rows_before = len(data)
  print("All columns with blanks:\n", blanks_before[blanks_before > 0])
count_blanks(ev_grid_og)

Blanks in Avg_Temp_F: 54
Blanks in Public_Charger_Count: 54
Blanks in load columns (should be 0):
 Base_Load_MW           0
Home_EV_Load_MW        0
Public_EV_Load_MW      0
EV_Charging_Load_MW    0
Total_Load_MW          0
dtype: int64
All columns with blanks:
 Public_Charger_Count    54
Avg_Temp_F              54
dtype: int64


In [27]:
typical_temp = ev_grid_working.groupby(["Month", "Time_Block"])["Avg_Temp_F"].transform("median")
ev_grid_working["Avg_Temp_F"] = ev_grid_working["Avg_Temp_F"].fillna(typical_temp).round(1)

typical_chargers = ev_grid_working.groupby(["Zone_ID", "Year", "Month"])["Public_Charger_Count"].transform("median")
ev_grid_working["Public_Charger_Count"] = ev_grid_working["Public_Charger_Count"].fillna(typical_chargers).round()

count_blanks(ev_grid_working)


Blanks in Avg_Temp_F: 0
Blanks in Public_Charger_Count: 0
Blanks in load columns (should be 0):
 Base_Load_MW           0
Home_EV_Load_MW        0
Public_EV_Load_MW      0
EV_Charging_Load_MW    0
Total_Load_MW          0
dtype: int64
All columns with blanks:
 Series([], dtype: int64)


In [28]:
def find_outliers(check_cols, data):
  outliers={}
  for col in check_cols:
    ratio = data[col] / check_cols[col]
    outliers[col] = ratio>1.2
    print(f"Total Outliers for {col} = {outliers[col].sum()}")
  return outliers

In [29]:
typical_base_load = ev_grid_working.groupby(["Zone_ID", "Month", "Time_Block"])["Base_Load_MW"].transform("median")
typical_home_load = ev_grid_working.groupby(["Zone_ID", "Year", "Time_Block", "Discount_Active"])["Home_EV_Load_MW"].transform("median")
typical_public_load = ev_grid_working.groupby(["Zone_ID", "Year", "Time_Block", "Discount_Active"])["Public_EV_Load_MW"].transform("median")

outlier_possible_cols = {"Base_Load_MW":typical_base_load, "Home_EV_Load_MW":typical_home_load, "Public_EV_Load_MW":typical_public_load}
outlier_rows = find_outliers(outlier_possible_cols, ev_grid_working)

log = []
for col in outlier_possible_cols:
    typicals = outlier_possible_cols[col]
    for i in ev_grid_working.index[outlier_rows[col]]:
        log.append({"Row_ID": ev_grid_working.at[i, "Row_ID"], "Column": col,
                    "Old_Value": ev_grid_working.at[i, col],
                    "New_Value": round(typicals[i], 3)})
    ev_grid_working.loc[outlier_rows[col], col] = typicals[outlier_rows[col]].round(3)

log_df = pd.DataFrame(log)
log_df.to_csv("outlier_log.csv", index=False)
print(len(log_df), "outliers fixed")

Total Outliers for Base_Load_MW = 46
Total Outliers for Home_EV_Load_MW = 82
Total Outliers for Public_EV_Load_MW = 97
225 outliers fixed


In [31]:
ev_grid_working["EV_Charging_Load_MW"] = (ev_grid_working["Home_EV_Load_MW"] + ev_grid_working["Public_EV_Load_MW"]).round(3)
ev_grid_working["Total_Load_MW"] = (ev_grid_working["Base_Load_MW"] + ev_grid_working["EV_Charging_Load_MW"]).round(3)
print(ev_grid_working)

      Row_ID Zone_ID      Zone_Name  Year  Month    Time_Block  Homes_in_Zone  \
0          1     Z01       Bellevue  2019      1       Morning           1500   
1          2     Z01       Bellevue  2019      1     Afternoon           1500   
2          3     Z01       Bellevue  2019      1  Evening_Peak           1500   
3          4     Z01       Bellevue  2019      1     Overnight           1500   
4          5     Z01       Bellevue  2019      2       Morning           1500   
...      ...     ...            ...   ...    ...           ...            ...   
5371    5372     Z16  Battle Ground  2025     11     Overnight            639   
5372    5373     Z16  Battle Ground  2025     12       Morning            639   
5373    5374     Z16  Battle Ground  2025     12     Afternoon            639   
5374    5375     Z16  Battle Ground  2025     12  Evening_Peak            639   
5375    5376     Z16  Battle Ground  2025     12     Overnight            639   

      EV_Share_Pct  Public_